In [9]:
import os
import sys
import subprocess
import tensorflow as tf

# 1. Tarik atau perbarui repositori terbaru dari GitHub (berlaku untuk semua user)
REPO_URL = "https://github.com/fa7ahillah/pneumonia-sota-classification.git"
REPO_NAME = "pneumonia-sota-classification"

BASE_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
PROJECT_ROOT = os.path.join(BASE_DIR, REPO_NAME)

if not os.path.exists(PROJECT_ROOT):
    print(f"Mengunduh repositori ke {PROJECT_ROOT}...")
    subprocess.run(["git", "clone", REPO_URL, PROJECT_ROOT], check=True)
else:
    print(f"Memperbarui repositori di {PROJECT_ROOT}...")
    subprocess.run(["git", "-C", PROJECT_ROOT, "pull", "origin", "main"], check=True)

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# 2. Mount Google Drive secara opsional dan aman (aman jika sudah ter-mount atau di Kaggle)
IS_COLAB = False
try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    else:
        print("Google Drive sudah ter-mount di /content/drive.")
    IS_COLAB = True
except ImportError:
    print("Berjalan di lingkungan non-Colab (misal: Kaggle/Lokal), melewati mount Drive.")
except ValueError as e:
    print(f"Peringatan mount Drive dilewati ({e}). Proyek tetap berjalan di {PROJECT_ROOT}.")
    IS_COLAB = True

# 3. Verifikasi direktori proyek & akselerator GPU T4
gpus = tf.config.list_physical_devices("GPU")
print("Direktori aktif  :", os.getcwd())
print("Isi direktori    :", os.listdir(PROJECT_ROOT))
print("GPU aktif        :", gpus)

if not gpus:
    print("⚠️ PERINGATAN: GPU tidak terdeteksi! Ubah Runtime/Accelerator ke GPU T4 sebelum melatih model.")

Memperbarui repositori di /content/pneumonia-sota-classification...
Peringatan mount Drive dilewati (Mountpoint must not already contain files). Proyek tetap berjalan di /content/pneumonia-sota-classification.
Direktori aktif  : /content/pneumonia-sota-classification
Isi direktori    : ['results', 'notebooks', 'src', '.gitignore', 'data', 'README.md', '.git']
GPU aktif        : []
⚠️ PERINGATAN: GPU tidak terdeteksi! Ubah Runtime/Accelerator ke GPU T4 sebelum melatih model.


In [10]:
import os
import sys
import importlib
from getpass import getpass

# Pastikan direktori proyek aktif diprioritaskan dan modul dimuat ulang
project_dir = os.getcwd()
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

importlib.invalidate_caches()
for mod_name in ["src", "src.config", "src.dataset"]:
    sys.modules.pop(mod_name, None)

from src import config, dataset, models, evaluate

# Ambil kredensial dari Colab Secrets (jika sudah disimpan) atau minta input aman saat runtime
try:
    from google.colab import userdata
    KAGGLE_USERNAME = userdata.get("KAGGLE_USERNAME")
    KAGGLE_KEY = userdata.get("KAGGLE_KEY")
except Exception:
    KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME") or input("Masukkan Kaggle Username: ")
    KAGGLE_KEY = os.environ.get("KAGGLE_KEY") or getpass("Masukkan Kaggle API Key: ")

# 1. Unduh & ekstrak dataset langsung dari Kaggle ke disk lokal VM Colab
dataset.download_kaggle_dataset(
    kaggle_username=KAGGLE_USERNAME,
    kaggle_key=KAGGLE_KEY,
)

# 2. Pembagian data seimbang (70:15:15) dengan SEED = 42
train_df, val_df, test_df = dataset.prepare_balanced_splits(force_regenerate=False)

print("Train :", len(train_df), train_df["label"].value_counts().to_dict())
print("Val   :", len(val_df), val_df["label"].value_counts().to_dict())
print("Test  :", len(test_df), test_df["label"].value_counts().to_dict())

[INFO] Mengunduh dataset 'pcbreviglieri/pneumonia-xray-images' dari Kaggle...
[INFO] Mengekstrak 'pneumonia-xray-images.zip' ke /content/pneumonia_workspace/raw...
[INFO] Ekstraksi selesai di: /content/pneumonia_workspace/raw
[INFO] Total Citra Asli -> NORMAL: 1583, PNEUMONIA: 4273
[INFO] File manifest CSV berhasil disimpan ke: /content/drive/MyDrive/DeepLearning/Pneumonia SoTA Classification/data/splits
Train : 1400 {'NORMAL': 700, 'PNEUMONIA': 700}
Val   : 300 {'PNEUMONIA': 150, 'NORMAL': 150}
Test  : 300 {'PNEUMONIA': 150, 'NORMAL': 150}


In [ ]:
import os
import sys
import random
import subprocess
import numpy as np
import pandas as pd
import tensorflow as tf

# --- 1. PASTIKAN DIREKTORI PROYEK & MODUL 'src' TERBACA ---
REPO_NAME = "pneumonia-sota-classification"
GITHUB_USER = "fa7ahillah"

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
elif not os.path.exists("src"):
    for candidate in [REPO_NAME, f"/content/{REPO_NAME}", f"/kaggle/working/{REPO_NAME}"]:
        if os.path.exists(candidate):
            os.chdir(candidate)
            break
    else:
        subprocess.run(
            ["git", "clone", f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"],
            check=True,
        )
        os.chdir(REPO_NAME)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from src import config, dataset, models, evaluate


# --- 2. FUNGSI SEED & PIPELINE PELATIHAN ---
def set_global_seed(seed=None):
    if seed is None:
        seed = config.SEED
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def run_experiment(arch_name, train_df, val_df, test_df, n_reps=None, ep1=10, ep2=20):
    if n_reps is None:
        n_reps = config.N_REPETITIONS

    records = []
    histories = []
    best_f1 = -1.0
    best_info = {}

    save_dir = os.path.join(config.SAVED_MODELS_DIR, arch_name)
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(config.LOGS_DIR, exist_ok=True)

    for run in range(1, n_reps + 1):
        print(f"\n{'='*60}\nMODEL: {arch_name} | REPETISI {run}/{n_reps}\n{'='*60}")
        tf.keras.backend.clear_session()
        set_global_seed(config.SEED)

        train_gen, val_gen, test_gen = dataset.create_data_generators(train_df, val_df, test_df)
        model, history, duration, p1, p2 = models.train_two_phase(
            arch_name, train_gen, val_gen, epochs_p1=ep1, epochs_p2=ep2
        )
        histories.append(history)

        metrics = evaluate.evaluate_on_test_set(model, test_gen)
        records.append({
            "Model": arch_name,
            "Repetisi": run,
            "Trainable_Params_Phase1": p1,
            "Trainable_Params_Phase2": p2,
            "Training_Time_Sec": round(duration, 2),
            "Accuracy": metrics["Accuracy"],
            "Precision": metrics["Precision"],
            "Recall": metrics["Recall"],
            "F1_Score": metrics["F1_Score"],
            "TP": metrics["TP"],
            "TN": metrics["TN"],
            "FP": metrics["FP"],
            "FN": metrics["FN"],
        })

        print(
            f"[Run {run}] Acc: {metrics['Accuracy']:.4f} | Prec: {metrics['Precision']:.4f} | "
            f"Rec: {metrics['Recall']:.4f} | F1: {metrics['F1_Score']:.4f} | "
            f"FP: {metrics['FP']} | FN: {metrics['FN']} | Time: {duration:.2f}s"
        )

        if metrics["F1_Score"] > best_f1:
            best_f1 = metrics["F1_Score"]
            model.save(os.path.join(save_dir, f"{arch_name}_best.keras"))
            model.save(os.path.join(save_dir, f"{arch_name}_best.h5"))
            best_info = {
                "arch_name": arch_name,
                "best_run": run,
                "accuracy": metrics["Accuracy"],
                "precision": metrics["Precision"],
                "recall": metrics["Recall"],
                "f1_score": metrics["F1_Score"],
                "cm": metrics["cm"],
            }

        pd.DataFrame(records).to_csv(
            os.path.join(config.LOGS_DIR, f"hasil_{arch_name}_progress.csv"),
            index=False,
        )

    return pd.DataFrame(records), histories, best_info


# --- 3. EKSEKUSI EKSPERIMEN ---
# Ubah N_REPS menjadi 10 untuk menjalankan protokol penuh sesuai proposal[cite: 5]
N_REPS = 1
EPOCHS_PHASE_1 = 10
EPOCHS_PHASE_2 = 20

# Jika Sel 2 belum dijalankan pada sesi ini, panggil fungsi dari src.dataset
if "train_df" not in globals():
    dataset.download_kaggle_dataset()
    train_df, val_df, test_df = dataset.prepare_balanced_splits(force_regenerate=False)

df_res, hist_res, best_res = run_experiment(
    "ResNet50", train_df, val_df, test_df, n_reps=N_REPS, ep1=EPOCHS_PHASE_1, ep2=EPOCHS_PHASE_2
)
df_mob, hist_mob, best_mob = run_experiment(
    "MobileNetV2", train_df, val_df, test_df, n_reps=N_REPS, ep1=EPOCHS_PHASE_1, ep2=EPOCHS_PHASE_2
)

all_results = pd.concat([df_res, df_mob], ignore_index=True)
output_csv = os.path.join(config.LOGS_DIR, f"hasil_eksperimen_sota_{N_REPS}run.csv")
all_results.to_csv(output_csv, index=False)
print(f"\nSeluruh hasil log tersimpan di: {output_csv}")

summary = evaluate.summarize_results(all_results)
print(f"\n=== RINGKASAN EVALUASI ({N_REPS} RUN PENUH) ===")
print(summary)

evaluate.plot_comparison_confusion_matrices(best_res, best_mob, all_results)
evaluate.plot_learning_curves(hist_res, "ResNet50", split_epoch=EPOCHS_PHASE_1)
evaluate.plot_learning_curves(hist_mob, "MobileNetV2", split_epoch=EPOCHS_PHASE_1)


MODEL: ResNet50 | REPETISI 1/1
Found 1400 validated image filenames belonging to 2 classes.
Found 300 validated image filenames belonging to 2 classes.
Found 300 validated image filenames belonging to 2 classes.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 453s 10s/step - accuracy: 0.7843 - loss: 0.4469 - val_accuracy: 0.9333 - val_loss: 0.1993
Epoch 2/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 451s 10s/step - accuracy: 0.8950 - loss: 0.2778 - val_accuracy: 0.9267 - val_loss: 0.1747
Epoch 3/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 434s 10s/step - accuracy: 0.9057 - loss: 0.2415 - val_accuracy: 0.9367 - val_loss: 0.1626
Epoch 4/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 426s 10s/step - accuracy: 0.9121 - loss: 0.2389 - val_accuracy: 0.9467 - val_loss: 0.1624
Epoch 5/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 440s 10s/step - accuracy: 0.9186 - loss: 0.2099 - val_accuracy: 0.9333 - val_loss: 0.1513
Epoch 6/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 455s 10s/step - accuracy: 0.9064 - loss: 0.2233 - val_accu